In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
path = open("C:\Projects\Spam\spam.csv")
df = pd.read_csv(path)

In [3]:
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)

In [4]:
df = df.rename(columns={'v1': 'statuss', 'v2': 'Words'})

In [5]:
head = df.head()
shape = df.shape
dtypes = df.dtypes

print(f'Head:\n {head}\nshape:\n {shape}\nData Types: \n{dtypes}')

Head:
   statuss                                              Words
0     ham  Go until jurong point, crazy.. Available only ...
1     ham                      Ok lar... Joking wif u oni...
2    spam  Free entry in 2 a wkly comp to win FA Cup fina...
3     ham  U dun say so early hor... U c already then say...
4     ham  Nah I don't think he goes to usf, he lives aro...
shape:
 (5572, 2)
Data Types: 
statuss    object
Words      object
dtype: object


In [6]:
# X = df.drop(columns='status')
y = df['statuss']

In [7]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['Words'])
X = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
X

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,0125698789,02,...,û_,û_thanks,ûªm,ûªt,ûªve,ûï,ûïharry,ûò,ûówell,žö
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5568,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5569,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [9]:
from sklearn.svm import LinearSVC

model = LinearSVC(C=1.0)


model.fit(X_train, y_train)

LinearSVC()

In [10]:
y_pred = model.predict(X_test)

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [12]:
# Basic accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9799

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      1587
        spam       0.98      0.87      0.92       252

    accuracy                           0.98      1839
   macro avg       0.98      0.93      0.96      1839
weighted avg       0.98      0.98      0.98      1839


Confusion Matrix:
[[1583    4]
 [  33  219]]


In [13]:
param_grid = {
    'C': [0.1, 1.0, 10.0],  # Regularization parameter
    'loss': ['hinge', 'squared_hinge'],  # Loss function
    'tol': [1e-4, 1e-3]  # Tolerance for stopping criteria
}
grid_search = GridSearchCV(
    model,
    {
        'C': [0.1, 1.0, 10.0, 20.0, 30.0, 40.0, 100.0],
    },
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,  # Use all processors
    verbose=2,  # Show progress
    scoring='accuracy'
)

In [14]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


GridSearchCV(cv=5, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1.0, 10.0, 20.0, 30.0, 40.0, 100.0]},
             scoring='accuracy', verbose=2)

In [15]:
# 7. Print results
print("\nBest parameters:")
print(grid_search.best_params_)
print(f"\nBest cross-validation score: {grid_search.best_score_:.4f}")


Best parameters:
{'C': 20.0}

Best cross-validation score: 0.9821


In [16]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [17]:
print("Baseline")
print("\nPerformance on test set:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Baseline

Performance on test set:
Accuracy: 0.9799

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      1587
        spam       0.98      0.87      0.92       252

    accuracy                           0.98      1839
   macro avg       0.98      0.93      0.96      1839
weighted avg       0.98      0.98      0.98      1839


Confusion Matrix:
[[1582    5]
 [  32  220]]


In [18]:
# Step 1: List of common spam words
spam_words = [
    'free', 'win', 'winner', 'prize', 'buy now', 'click', 'cash', 
    'offer', 'rich', 'money', 'urgent', 'claim', 'limited', 'act now',
    'guarantee', 'credit', 'cheap', 'discount', 'deal', '!'
]


# Step 2: Function to count spam words in each text
def count_spam_words(text):
    text = text.lower()
    return sum(1 for word in spam_words if word in text)


# Step 3: Apply both features to DataFrame
df['spam_word_count'] = df['Words'].apply(count_spam_words)
df['message_length'] = df['Words'].apply(len)  # Total character count

# Optional: word count too
df['word_count'] = df['Words'].apply(lambda x: len(x.split()))

# Show the updated DataFrame
df_FE = pd.DataFrame(df)
df_FE.head(3)

,statuss,Words,spam_word_count,message_length,word_count
0,ham,"Go until jurong point, crazy.. Available only ...",0,111,20
1,ham,Ok lar... Joking wif u oni...,0,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,2,155,28


In [19]:
import joblib

In [20]:
# Step 1: Vectorize the 'Words' column
vectorize_words = vectorizer.fit_transform(df_FE['Words'])

# Step 2: Convert to DataFrame with TF-IDF features
vectorize_words = pd.DataFrame(
    vectorize_words.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Step 3: Select other features from original DataFrame
new_df = df_FE[['statuss', 'spam_word_count', 'message_length', 'word_count']].reset_index(drop=True)

# Step 4: Concatenate TF-IDF features with the selected columns
new_df = pd.concat([vectorize_words, new_df], axis=1)
new_df.head()

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [21]:
a = new_df.drop(columns='statuss')
r = new_df['statuss']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(a, r, test_size=0.33, random_state=42)

In [23]:
model = LinearSVC(C=1.0)

param_grid = {
    'C': [0.1, 1.0, 10.0],  # Regularization parameter
    'loss': ['hinge', 'squared_hinge'],  # Loss function
    'tol': [1e-4, 1e-3]  # Tolerance for stopping criteria
}
grid_search = GridSearchCV(
    model,
    {
        'C': [0.1, 1.0, 10.0, 20.0, 30.0, 40.0, 100.0],
    },
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,  # Use all processors
    verbose=2,  # Show progress
    scoring='accuracy'
)

In [24]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


GridSearchCV(cv=5, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1.0, 10.0, 20.0, 30.0, 40.0, 100.0]},
             scoring='accuracy', verbose=2)

In [25]:
# 7. Print results
print("\nBest parameters:")
print(grid_search.best_params_)
print(f"\nBest cross-validation score: {grid_search.best_score_:.4f}")


Best parameters:
{'C': 20.0}

Best cross-validation score: 0.9435


In [26]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [27]:


joblib.dump(best_model, 'spam_classifier_model.pkl')

['spam_classifier_model.pkl']

In [28]:
print("Baseline")
print("\nPerformance on test set:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Baseline

Performance on test set:
Accuracy: 0.9239

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      0.93      0.95      1587
        spam       0.67      0.88      0.76       252

    accuracy                           0.92      1839
   macro avg       0.82      0.91      0.86      1839
weighted avg       0.94      0.92      0.93      1839


Confusion Matrix:
[[1477  110]
 [  30  222]]


In [29]:
input = input("Enter Email Content:\n")

df = pd.DataFrame([[input]], columns=['Words'])


# pre-process input data

# Step 1: List of common spam words
spam_words = [
    'free', 'win', 'winner', 'prize', 'buy now', 'click', 'cash', 
    'offer', 'rich', 'money', 'urgent', 'claim', 'limited', 'act now',
    'guarantee', 'credit', 'cheap', 'discount', 'deal', '!'
]


# Step 2: Function to count spam words in each text
def count_spam_words(text):
    text = text.lower()
    return sum(1 for word in spam_words if word in text)


# Step 3: Apply both features to DataFrame
df['spam_word_count'] = df['Words'].apply(count_spam_words)
df['message_length'] = df['Words'].apply(len)  # Total character count

# Optional: word count too
df['word_count'] = df['Words'].apply(lambda x: len(x.split()))

# Show the updated DataFrame
df_FE = pd.DataFrame(df)

vectorizer = TfidfVectorizer()

# Step 1: Vectorize the 'Words' column
vectorize_words = vectorizer.fit_transform(df_FE['Words'])

# Step 2: Convert to DataFrame with TF-IDF features
vectorize_words = pd.DataFrame(
    vectorize_words.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Step 3: Select other features from original DataFrame
new_df = df_FE[['spam_word_count', 'message_length', 'word_count']].reset_index(drop=True)

# Step 4: Concatenate TF-IDF features with the selected columns
new_df = pd.concat([vectorize_words, new_df], axis=1)

pred = best_model.predict(new_df)

vectorize_words

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- 00
- 000
- 000pes
- 008704050406
- 0089
- ...
